In [ ]:
from pyqueueing import ErlangC, ErlangB, MMC

## Scenario

A call center receives **100 calls per hour**. Each call takes **5 minutes** on average.

- Arrival rate λ = 100 calls/hour
- Service rate μ = 60/5 = 12 calls/hour per agent
- Offered load a = λ/μ ≈ 8.33 Erlangs

In [ ]:
lam = 100   # calls/hour
mu = 12     # calls/hour per agent
print(f"Offered load: {lam/mu:.2f} Erlangs")

## 1. Required Agents — Three Approaches

In [ ]:
ec = ErlangC(arrival_rate=lam, service_rate=mu)

# Approach 1: Target wait probability
c1 = ec.required_servers(target_wait_prob=0.20)
print(f"Target P(wait) ≤ 20%:    {c1} agents")

# Approach 2: Target mean wait time (30 seconds = 30/3600 hours)
c2 = ec.required_servers(target_mean_wait=30/3600)
print(f"Target E[Wq] ≤ 30 sec:   {c2} agents")

# Approach 3: Service Level — 80% answered within 20 seconds
c3 = ec.required_servers(target_service_level=(20/3600, 0.80))
print(f"Target 80/20 SL:         {c3} agents")

## 2. Performance at Different Staffing Levels

In [ ]:
print(f"{'Agents':>6} {'P(wait)':>8} {'E[Wq] sec':>10} {'SL 80/20':>9}")
print("-" * 38)

for c in range(9, 15):
    mmc = MMC(arrival_rate=lam, service_rate=mu, servers=c)
    pw = mmc.prob_wait()
    wq_sec = mmc.mean_wait() * 3600  # convert hours to seconds
    sl = mmc.wait_time_cdf(20/3600)  # P(Wq ≤ 20 sec)
    print(f"{c:>6d} {pw:>8.2%} {wq_sec:>10.1f} {sl:>9.2%}")

## 3. Erlang B — Trunk Line Provisioning

If there is **no waiting room** (calls get a busy signal), how many lines do we need?

In [ ]:
eb = ErlangB(arrival_rate=lam, service_rate=mu, servers=1)
trunks = eb.required_servers(target_block_prob=0.01)
print(f"Required lines for <1% blocking: {trunks}")

# Verify
eb2 = ErlangB(arrival_rate=lam, service_rate=mu, servers=trunks)
print(f"Actual blocking rate: {eb2.prob_block():.4%}")

## 4. Comparing Erlang B vs Erlang C

Erlang B (loss): caller gets busy signal → **blocking probability**

Erlang C (delay): caller waits in queue → **wait probability**

In [ ]:
from pyqueueing.models.erlang_b import erlang_b_formula
from pyqueueing.models.mmc import _erlang_c

a = lam / mu
print(f"{'Servers':>7} {'Erlang B':>10} {'Erlang C':>10}")
print("-" * 30)
for c in range(9, 16):
    eb = erlang_b_formula(c, a)
    ec_val = _erlang_c(c, a)
    print(f"{c:>7d} {eb:>10.4%} {ec_val:>10.4%}")